# Cuaderno del proyecto, en el proyecto se pide un cuaderno que haga todo y un cuaderno que explique los resultados pero puede ser todo en el mismo cuadrno, se pide la base de datos, un txt con los requerimientos para correr el cuaderno y ya

# 1. Dataset, lector y verificaciones:

## 1.1 Imports, descarga, extracción y lectura previa de datos:

In [39]:
#Imports 
import time
import requests
import re
import sqlite3
from datetime import datetime, timezone


#Definir la descarga e intentos
def download_file(url, destination, max_retries=3):
    """
    Download a file using streaming and automatic retries.

    Parameters
    ----------
    url : str
        URL of the file to download.

    destination : pathlib.Path
        Local path where the file will be saved.

    max_retries : int
        Maximum number of download attempts.
    """

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 "
            "Chrome/149.0 Safari/537.36"
        )
    }

    for attempt in range(1, max_retries + 1):

        try:
            print(f"Download attempt {attempt}/{max_retries}...")

            with requests.get(
                url,
                headers=headers,
                stream=True,
                timeout=(30, 300)
            ) as response:

                response.raise_for_status()

                total_size = int(
                    response.headers.get("content-length", 0)
                )

                downloaded = 0

                with open(destination, "wb") as file:

                    for chunk in response.iter_content(
                        chunk_size=1024 * 1024
                    ):
                        if chunk:
                            file.write(chunk)
                            downloaded += len(chunk)

                            if total_size > 0:
                                progress = 100 * downloaded / total_size

                                print(
                                    f"\rDownloading: {progress:6.2f}%",
                                    end=""
                                )

            print("\nDownload completed successfully.")
            print(f"Saved to: {destination}")

            return

        except requests.exceptions.RequestException as error:

            print(f"\nDownload attempt failed: {error}")

            # Delete an incomplete download
            if destination.exists():
                destination.unlink()

            if attempt == max_retries:
                raise RuntimeError(
                    "Failed to download the SPARC dataset "
                    f"after {max_retries} attempts."
                ) from error

            print("Retrying in 5 seconds...")
            time.sleep(5)


#Descarga del ZIP
if ZIP_PATH.exists():
    print(f"ZIP file already exists: {ZIP_PATH}")

else:
    print("Downloading SPARC rotation-curve data...")

    download_file(
        SPARC_URL,
        ZIP_PATH,
        max_retries=3
    )


#Extracción del ZIP 
dat_files = list(EXTRACT_DIR.rglob("*.dat"))

if dat_files:
    print("Dataset has already been extracted.")
else:
    print("Extracting SPARC data...")

    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)

    print("Extraction completed successfully.")


#Verificación de datos descargados
dat_files = sorted(EXTRACT_DIR.rglob("*.dat"))

if len(dat_files) == 0:
    raise FileNotFoundError(
        "No .dat files were found after extracting the ZIP archive."
    )

print(f"Number of .dat files found: {len(dat_files)}")

print("\nFirst five files:")
for file_path in dat_files[:5]:
    print(file_path.name)


#Inspeccionar archivos 
sample_file = dat_files[0]

print(f"Inspecting file: {sample_file.name}")
print("-" * 70)

with open(sample_file, "r") as file:
    for _ in range(15):
        line = file.readline()

        if not line:
            break

        print(line.rstrip())

ZIP file already exists: data/raw/Rotmod_LTG.zip
Dataset has already been extracted.
Number of .dat files found: 175

First five files:
CamB_rotmod.dat
D512-2_rotmod.dat
D564-8_rotmod.dat
D631-7_rotmod.dat
DDO064_rotmod.dat
Inspecting file: CamB_rotmod.dat
----------------------------------------------------------------------
# Distance = 3.36 Mpc
# Rad	Vobs	errV	Vgas	Vdisk	Vbul	SBdisk	SBbul
# kpc	km/s	km/s	km/s	km/s	km/s	L/pc^2	L/pc^2
0.16	1.99	1.50	1.86	3.75	0.00	30.32	0.00
0.41	4.84	1.50	4.24	9.47	0.00	23.77	0.00
0.57	6.79	1.50	5.61	11.76	0.00	15.87	0.00
0.73	8.87	1.50	6.77	13.72	0.00	12.40	0.00
0.90	10.90	1.50	7.77	14.80	0.00	9.63	0.00
1.06	12.90	1.50	8.44	15.24	0.00	5.86	0.00
1.22	14.70	1.50	8.64	15.11	0.00	5.19	0.00
1.47	16.80	1.50	8.08	15.90	0.00	3.02	0.00
1.79	20.10	1.50	6.91	14.91	0.00	0.88	0.00


## 1.2 Construcción del lector de datos:

In [32]:
#Dedfinimos las columnas a utilizar 
COLUMN_NAMES = [
    "radius_kpc",
    "v_obs_kms",
    "err_v_kms",
    "v_gas_kms",
    "v_disk_kms",
    "v_bulge_kms",
    "sb_disk_l_pc2",
    "sb_bulge_l_pc2",
]

#Definimos el lector 
def read_sparc_rotation_curve(file_path):
    """
    Read one SPARC Newtonian mass-model file.

    The function extracts:
    - Galaxy name from the file name.
    - Galaxy distance from the first header line.
    - Rotation-curve measurements from the numerical table.

    Parameters
    ----------
    file_path : str or pathlib.Path
        Path to a SPARC *_rotmod.dat file.

    Returns
    -------
    galaxy_info : dict
        Basic information about the galaxy.

    rotation_curve : pandas.DataFrame
        Rotation-curve measurements.
    """

    file_path = Path(file_path)

    # Extract galaxy name from file name
    galaxy_name = file_path.name.replace("_rotmod.dat", "")

    # Read the first line to extract the distance
    with open(file_path, "r") as file:
        first_line = file.readline().strip()

    distance_match = re.search(
        r"Distance\s*=\s*([\d.]+)\s*Mpc",
        first_line
    )

    if distance_match is None:
        raise ValueError(
            f"Could not extract distance from {file_path.name}"
        )

    distance_mpc = float(distance_match.group(1))

    # Read numerical rotation-curve data
    rotation_curve = pd.read_csv(
        file_path,
        sep=r"\s+",
        comment="#",
        names=COLUMN_NAMES
    )

    # Add galaxy name for traceability
    rotation_curve.insert(
        0,
        "galaxy",
        galaxy_name
    )

    galaxy_info = {
        "galaxy": galaxy_name,
        "distance_mpc": distance_mpc,
        "n_points": len(rotation_curve)
    }

    return galaxy_info, rotation_curve

#Prueba del lector (individual) 
sample_info, sample_df = read_sparc_rotation_curve(sample_file)

print("Galaxy information:")
print(sample_info)

print("\nRotation curve:")
display(sample_df.head())

#Prueba de lector (poblacional)
galaxy_records = []
rotation_curve_frames = []

for file_path in dat_files:

    galaxy_info, galaxy_curve = read_sparc_rotation_curve(
        file_path
    )

    galaxy_records.append(galaxy_info)
    rotation_curve_frames.append(galaxy_curve)

#Construimos los dataframes 
galaxies_df = pd.DataFrame(galaxy_records)

rotation_curves_df = pd.concat(
    rotation_curve_frames,
    ignore_index=True
)

#Comprobación del dataframe
print("SPARC dataset successfully loaded.")
print("-" * 50)

print(
    f"Number of galaxies: "
    f"{galaxies_df['galaxy'].nunique()}"
)

print(
    f"Total rotation-curve measurements: "
    f"{len(rotation_curves_df)}"
)

print("\nGalaxy table:")
display(galaxies_df.head())

print("\nRotation-curve table:")
display(rotation_curves_df.head())

Galaxy information:
{'galaxy': 'CamB', 'distance_mpc': 3.36, 'n_points': 9}

Rotation curve:


,galaxy,radius_kpc,v_obs_kms,err_v_kms,v_gas_kms,v_disk_kms,v_bulge_kms,sb_disk_l_pc2,sb_bulge_l_pc2
0,CamB,0.16,1.99,1.5,1.86,3.75,0.0,30.32,0.0
1,CamB,0.41,4.84,1.5,4.24,9.47,0.0,23.77,0.0
2,CamB,0.57,6.79,1.5,5.61,11.76,0.0,15.87,0.0
3,CamB,0.73,8.87,1.5,6.77,13.72,0.0,12.40,0.0
4,CamB,0.90,10.90,1.5,7.77,14.80,0.0,9.63,0.0


SPARC dataset successfully loaded.
--------------------------------------------------
Number of galaxies: 175
Total rotation-curve measurements: 3391

Galaxy table:


,galaxy,distance_mpc,n_points
0,CamB,3.36,9
1,D512-2,15.20,4
2,D564-8,8.79,6
3,D631-7,7.72,16
4,DDO064,6.80,14



Rotation-curve table:


,galaxy,radius_kpc,v_obs_kms,err_v_kms,v_gas_kms,v_disk_kms,v_bulge_kms,sb_disk_l_pc2,sb_bulge_l_pc2
0,CamB,0.16,1.99,1.5,1.86,3.75,0.0,30.32,0.0
1,CamB,0.41,4.84,1.5,4.24,9.47,0.0,23.77,0.0
2,CamB,0.57,6.79,1.5,5.61,11.76,0.0,15.87,0.0
3,CamB,0.73,8.87,1.5,6.77,13.72,0.0,12.40,0.0
4,CamB,0.90,10.90,1.5,7.77,14.80,0.0,9.63,0.0


## 1.3 Control de calidad y limpieza de datos:

In [33]:
#Estructura y tipo
print("GALAXIES DATAFRAME")
print("-" * 50)
galaxies_df.info()

print("\nROTATION CURVES DATAFRAME")
print("-" * 50)
rotation_curves_df.info()

GALAXIES DATAFRAME
--------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 175 entries, 0 to 174
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   galaxy        175 non-null    str    
 1   distance_mpc  175 non-null    float64
 2   n_points      175 non-null    int64  
dtypes: float64(1), int64(1), str(1)
memory usage: 5.5 KB

ROTATION CURVES DATAFRAME
--------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 3391 entries, 0 to 3390
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   galaxy          3391 non-null   str    
 1   radius_kpc      3391 non-null   float64
 2   v_obs_kms       3391 non-null   float64
 3   err_v_kms       3391 non-null   float64
 4   v_gas_kms       3391 non-null   float64
 5   v_disk_kms      3391 non-null   float64
 6   v_bulge_kms     3391 non-n

In [34]:
#Valores faltantes 
print("Missing values in galaxies_df:")
print(galaxies_df.isna().sum())

print("\nMissing values in rotation_curves_df:")
print(rotation_curves_df.isna().sum())

Missing values in galaxies_df:
galaxy          0
distance_mpc    0
n_points        0
dtype: int64

Missing values in rotation_curves_df:
galaxy            0
radius_kpc        0
v_obs_kms         0
err_v_kms         0
v_gas_kms         0
v_disk_kms        0
v_bulge_kms       0
sb_disk_l_pc2     0
sb_bulge_l_pc2    0
dtype: int64


In [35]:
#Valores duplicados
print(
    "Duplicated galaxy rows:",
    galaxies_df.duplicated().sum()
)

print(
    "Duplicated rotation-curve rows:",
    rotation_curves_df.duplicated().sum()
)

Duplicated galaxy rows: 0
Duplicated rotation-curve rows: 0


In [36]:
#Validaciones físicas 
quality_checks = {
    "Non-positive distances":
        (galaxies_df["distance_mpc"] <= 0).sum(),

    "Non-positive radii":
        (rotation_curves_df["radius_kpc"] <= 0).sum(),

    "Non-positive velocity uncertainties":
        (rotation_curves_df["err_v_kms"] <= 0).sum(),

    "Negative observed velocities":
        (rotation_curves_df["v_obs_kms"] < 0).sum(),

    "Negative disk surface brightness":
        (rotation_curves_df["sb_disk_l_pc2"] < 0).sum(),

    "Negative bulge surface brightness":
        (rotation_curves_df["sb_bulge_l_pc2"] < 0).sum(),
}

quality_checks_df = pd.DataFrame(
    quality_checks.items(),
    columns=["Check", "Number of affected rows"]
)

display(quality_checks_df)

,Check,Number of affected rows
0,Non-positive distances,0
1,Non-positive radii,0
2,Non-positive velocity uncertainties,0
3,Negative observed velocities,0
4,Negative disk surface brightness,0
5,Negative bulge surface brightness,0


In [37]:
#Consistencia entre dataframes y consistencia de n-puntos 
galaxies_metadata = set(galaxies_df["galaxy"])
galaxies_curves = set(rotation_curves_df["galaxy"])

missing_in_curves = galaxies_metadata - galaxies_curves
missing_in_metadata = galaxies_curves - galaxies_metadata

print(
    "Galaxies without rotation-curve data:",
    missing_in_curves
)

print(
    "Rotation curves without galaxy metadata:",
    missing_in_metadata
)

#n-points (no más puntos que filas)
actual_counts = (
    rotation_curves_df
    .groupby("galaxy")
    .size()
    .rename("actual_n_points")
    .reset_index()
)

count_check = galaxies_df.merge(
    actual_counts,
    on="galaxy",
    how="left"
)

count_check["consistent"] = (
    count_check["n_points"]
    == count_check["actual_n_points"]
)

print(
    "Galaxies with inconsistent point counts:",
    (~count_check["consistent"]).sum()
)

Galaxies without rotation-curve data: set()
Rotation curves without galaxy metadata: set()
Galaxies with inconsistent point counts: 0


In [38]:
#Resumen de comprobaciones
acquisition_summary = pd.DataFrame({
    "Metric": [
        "Number of galaxies",
        "Total rotation-curve measurements",
        "Minimum distance [Mpc]",
        "Maximum distance [Mpc]",
        "Minimum points per galaxy",
        "Maximum points per galaxy",
    ],
    "Value": [
        len(galaxies_df),
        len(rotation_curves_df),
        galaxies_df["distance_mpc"].min(),
        galaxies_df["distance_mpc"].max(),
        galaxies_df["n_points"].min(),
        galaxies_df["n_points"].max(),
    ]
})

display(acquisition_summary)

,Metric,Value
0,Number of galaxies,175.00
1,Total rotation-curve measurements,3391.00
2,Minimum distance [Mpc],0.98
3,Maximum distance [Mpc],127.80
4,Minimum points per galaxy,4.00
5,Maximum points per galaxy,115.00


# 2. Diseño de la base de datos:

## 2.1 Importar SQL y definir la ruta:

In [40]:
#Crear la carpeta y definir la base
DATABASE_DIR = Path("database")
DATABASE_DIR.mkdir(parents=True, exist_ok=True)

DATABASE_PATH = DATABASE_DIR / "sparc_project.db"

print(f"Database path: {DATABASE_PATH.resolve()}")

Database path: /home/ronal/Computational-Astrophysics/SQL_Sparc_Project/database/sparc_project.db


In [41]:
#Creamos la conexión 
conn = sqlite3.connect(DATABASE_PATH)

# Enable foreign-key constraint enforcement
conn.execute("PRAGMA foreign_keys = ON;")

cursor = conn.cursor()

print("SQLite connection established.")

SQLite connection established.


## 2.2 Creación de tablas: 

In [43]:
#Crear la tabla galaxies
cursor.execute("""
CREATE TABLE IF NOT EXISTS galaxies (
    galaxy_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL UNIQUE,
    distance_mpc REAL NOT NULL CHECK (distance_mpc > 0),
    n_points INTEGER NOT NULL CHECK (n_points > 0)
);
""")

#Crear la tabla rotation_curves
cursor.execute("""
CREATE TABLE IF NOT EXISTS rotation_curves (
    point_id INTEGER PRIMARY KEY AUTOINCREMENT,

    galaxy_id INTEGER NOT NULL,

    radius_kpc REAL NOT NULL CHECK (radius_kpc > 0),
    v_obs_kms REAL NOT NULL,
    err_v_kms REAL NOT NULL CHECK (err_v_kms > 0),

    v_gas_kms REAL NOT NULL,
    v_disk_kms REAL NOT NULL,
    v_bulge_kms REAL NOT NULL,

    sb_disk_l_pc2 REAL NOT NULL,
    sb_bulge_l_pc2 REAL NOT NULL,

    FOREIGN KEY (galaxy_id)
        REFERENCES galaxies(galaxy_id)
        ON DELETE CASCADE
);
""")

#Crear la tabla metadata
cursor.execute("""
CREATE TABLE IF NOT EXISTS metadata (
    key TEXT PRIMARY KEY,
    value TEXT NOT NULL
);
""")

#Creación de índices para consultas: 
cursor.execute("""
CREATE INDEX IF NOT EXISTS idx_rotation_curves_galaxy
ON rotation_curves(galaxy_id);
""")
cursor.execute("""
CREATE INDEX IF NOT EXISTS idx_rotation_curves_radius
ON rotation_curves(radius_kpc);
""")
conn.commit()

#Comprobación final
print("Database schema created successfully.")

Database schema created successfully.


## 2.3 Insertamos galaxias y puntos: 

In [44]:
galaxy_records = list(
    galaxies_df[
        [
            "galaxy",
            "distance_mpc",
            "n_points"
        ]
    ].itertuples(
        index=False,
        name=None
    )
)

cursor.executemany("""
INSERT OR IGNORE INTO galaxies (
    name,
    distance_mpc,
    n_points
)
VALUES (?, ?, ?);
""", galaxy_records)

conn.commit()

print("Galaxy data inserted successfully.")

#Pequeña comprobación 
n_galaxies_db = cursor.execute("""
SELECT COUNT(*)
FROM galaxies;
""").fetchone()[0]

print(f"Galaxies stored in database: {n_galaxies_db}")

Galaxy data inserted successfully.
Galaxies stored in database: 175


In [ ]:
#Ahora sucede que en el dataframe usamos CamB, D512-2, D564-8, ... , pero en el SQLite queremos guardar los puntos utilizando galaxy_id así que necesitamos crear una correspondencia primero

galaxy_id_rows = cursor.execute("""
SELECT galaxy_id, name
FROM galaxies;
""").fetchall()

galaxy_id_map = {
    name: galaxy_id
    for galaxy_id, name in galaxy_id_rows
}

list(galaxy_id_map.items())[:5]

[('CamB', 1), ('D512-2', 2), ('D564-8', 3), ('D631-7', 4), ('DDO064', 5)]

In [47]:
#Creamos una copia para no modificar el dataframe original
rotation_curves_sql = rotation_curves_df.copy()

#Añadimos el ID correspondiente
rotation_curves_sql["galaxy_id"] = (
    rotation_curves_sql["galaxy"]
    .map(galaxy_id_map)
)

#Verificamos que no haya galaxias sin ID 
missing_ids = (
    rotation_curves_sql["galaxy_id"]
    .isna()
    .sum()
)

print(f"Rotation-curve rows without galaxy_id: {missing_ids}")

Rotation-curve rows without galaxy_id: 0


In [49]:
#Agregamos los puntos ahora seleccionando las columnas en el orden exacto de la consulta SQL
rotation_records = list(
    rotation_curves_sql[
        [
            "galaxy_id",
            "radius_kpc",
            "v_obs_kms",
            "err_v_kms",
            "v_gas_kms",
            "v_disk_kms",
            "v_bulge_kms",
            "sb_disk_l_pc2",
            "sb_bulge_l_pc2",
        ]
    ].itertuples(
        index=False,
        name=None
    )
)

#Insertamos
cursor.executemany("""
INSERT INTO rotation_curves (
    galaxy_id,
    radius_kpc,
    v_obs_kms,
    err_v_kms,
    v_gas_kms,
    v_disk_kms,
    v_bulge_kms,
    sb_disk_l_pc2,
    sb_bulge_l_pc2
)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?);
""", rotation_records)

conn.commit()
print("Rotation-curve data inserted successfully.")

#Validación
n_points_db = cursor.execute("""
SELECT COUNT(*)
FROM rotation_curves;
""").fetchone()[0]

print(f"Rotation-curve measurements stored: {n_points_db}")

Rotation-curve data inserted successfully.
Rotation-curve measurements stored: 6782


In [ ]:
#AJÁAAAAAAAAAAAAAAAAAAA, ERROR, SON 3391 DATOS, CADA VEZ QUE EJECUTE LA CELDA SE INSERTAN MÁS DATOS, CORREGIR ESTE ERROR 